In [ ]:
!pip install -q transformers datasets scikit-learn


In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay


In [ ]:
columns = [
    "id", "label", "statement", "subject", "speaker", "speaker_job_title", "state_info",
    "party_affiliation", "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

train_df = pd.read_csv("train.tsv", sep="\t", header=None, names=columns)
valid_df = pd.read_csv("valid.tsv", sep="\t", header=None, names=columns)
test_df  = pd.read_csv("test.tsv",  sep="\t", header=None, names=columns)


In [ ]:

# Apply binary label mapping: Fake (0) vs Real (1)
label_map = {
    'pants-fire': 0,
    'false': 0,
    'barely-true': 0,
    'half-true': 1,
    'mostly-true': 1,
    'true': 1
}

for df in [train_df, valid_df, test_df]:
    df['label'] = df['label'].map(label_map)

# Drop rows with missing label mappings
train_df = train_df.dropna()
valid_df = valid_df.dropna()
test_df = test_df.dropna()

# Check label distribution
print("Train label counts:", train_df['label'].value_counts())
print("Validation label counts:", valid_df['label'].value_counts())
print("Test label counts:", test_df['label'].value_counts())


Train label counts: label
1    3929
0    2792
Name: count, dtype: int64
Validation label counts: label
1    472
0    389
Name: count, dtype: int64
Test label counts: label
1    504
0    349
Name: count, dtype: int64


In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def tokenize_fn(example):
    return tokenizer(example["statement"], padding="max_length", truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
valid_ds = Dataset.from_pandas(valid_df).map(tokenize_fn, batched=True)
test_ds  = Dataset.from_pandas(test_df).map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
valid_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/6721 [00:00<?, ? examples/s]

Map:   0%|          | 0/861 [00:00<?, ? examples/s]

Map:   0%|          | 0/853 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(pred):
    import numpy as np
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score

    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0, 1])
    acc_fake = accuracy_score(labels[labels == 0], preds[labels == 0])
    acc_true = accuracy_score(labels[labels == 1], preds[labels == 1])
    acc_overall = accuracy_score(labels, preds)

    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average='weighted')

    return {
        'accuracy_fake': acc_fake,
        'precision_fake': precision[0],
        'recall_fake': recall[0],
        'f1_fake': f1[0],
        'accuracy_true': acc_true,
        'precision_true': precision[1],
        'recall_true': recall[1],
        'f1_true': f1[1],
        'accuracy_weighted': acc_overall,
        'precision_weighted': precision_w,
        'recall_weighted': recall_w,
        'f1_weighted': f1_w
    }



In [ ]:
pip install --upgrade transformers


In [ ]:
!pip install --upgrade transformers --no-cache-dir


In [ ]:
import transformers
print(transformers.__version__)


4.51.3


In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids

    acc = accuracy_score(labels, preds)
    print(f"Computed metrics: accuracy={acc}")
    return {
        "accuracy": acc,
        # add more if needed like "f1": f1_score(labels, preds), etc.
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="steps",  # ✅ This existed in 4.5.1
    save_steps=500,               # 🔁 Save every 500 steps (adjust as needed)
    eval_steps=500,               # 🔎 Evaluate every 500 steps (adjust as needed)
    logging_dir="./logs",         # 📝 Required if logging is used
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",  # You may need to ensure your compute_metrics returns "accuracy"
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
500,0.514700,0.735450,0.626016


Computed metrics: accuracy=0.6260162601626016


TrainOutput(global_step=842, training_loss=0.5608030713369048, metrics={'train_runtime': 622.7762, 'train_samples_per_second': 21.584, 'train_steps_per_second': 1.352, 'total_flos': 1768369403074560.0, 'train_loss': 0.5608030713369048, 'epoch': 2.0})

In [ ]:
# Step 1: Run evaluation on test set
predictions = trainer.predict(test_ds)

# Step 2: Extract true and predicted labels
y_pred = predictions.predictions.argmax(axis=1)  # if logits
y_true = predictions.label_ids


Computed metrics: accuracy=0.6436107854630715


In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Step 3: Generate classification report
report = classification_report(y_true, y_pred, output_dict=True)

# Format as table
df = pd.DataFrame(report).transpose()
df['Model'] = 'RoBERTa'
df.reset_index(inplace=True)
df.rename(columns={'index': 'Label'}, inplace=True)
accuracy = report['accuracy']
df['Accuracy'] = accuracy

# Replace 'Fake' and 'True' with your actual labels if they're '0' and '1'
df = df[df['Label'].isin(['0', '1', 'weighted avg'])]  # or ['Fake', 'True', ...]
df = df[['Model', 'Label', 'Accuracy', 'precision', 'recall', 'f1-score']]
df.columns = ['Model', 'Label', 'Accuracy', 'Precision', 'Recall', 'F1 Score']
df[['Accuracy', 'Precision', 'Recall', 'F1 Score']] = df[['Accuracy', 'Precision', 'Recall', 'F1 Score']].round(4)

# Step 4: Print or save
print(df.to_string(index=False))


  Model        Label  Accuracy  Precision  Recall  F1 Score
RoBERTa            0    0.6436     0.5934  0.4097    0.4847
RoBERTa            1    0.6436     0.6634  0.8056    0.7276
RoBERTa weighted avg    0.6436     0.6347  0.6436    0.6282
